In [1]:
# ========================================================
# Importat API e bibliotecas necessárias para 
# começar a análise de dados de futebol internacional.
# ========================================================

import kagglehub as kh
import pandas as pd
import os
from joblib import dump, load

# 1. Download e Carga
path = kh.dataset_download("martj42/international-football-results-from-1872-to-2017")
df_results = pd.read_csv(f"{path}/results.csv")
df_shootouts = pd.read_csv(f"{path}/shootouts.csv")

# 2. União e Filtro 2014+
df_unido = pd.merge(df_results, df_shootouts[['date', 'home_team', 'away_team', 'winner']], 
                     on=['date', 'home_team', 'away_team'], how='left')

df_unido['date'] = pd.to_datetime(df_unido['date'])
df_final = df_unido[df_unido['date'].dt.year >= 2014].copy()
df_final = df_final.drop(columns=['city', 'country'])

# 3. Separar Passado (Treino) e Futuro (NaN)
df_treino = df_final.dropna(subset=['home_score']).copy()
df_copa_2026 = df_final[df_final['home_score'].isna()].copy() 

print(f"Treino: {df_treino.shape[0]} | Futuro (NaN): {df_copa_2026.shape[0]}")

c:\Users\COMERCIALL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Treino: 11842 | Futuro (NaN): 72


In [2]:
# ==============================================
# Criar uma função para medir a "forma" do time 
# (média dos últimos 10 jogos)
# ===============================================

def calcular_media_gols_10(team, date, df_completo):
    past_games = df_completo[((df_completo['home_team'] == team) | 
                              (df_completo['away_team'] == team)) & 
                             (df_completo['date'] < date)].sort_values('date', ascending=False).head(10)
    if past_games.empty: return 0
    gols = [row['home_score'] if row['home_team'] == team else row['away_score'] for _, row in past_games.iterrows()]
    return sum(gols) / len(gols)

# Aplicando as médias (Aguarde o processamento)
df_treino['home_form'] = df_treino.apply(lambda x: calcular_media_gols_10(x['home_team'], x['date'], df_treino), axis=1)
df_treino['away_form'] = df_treino.apply(lambda x: calcular_media_gols_10(x['away_team'], x['date'], df_treino), axis=1)

df_copa_2026['home_form'] = df_copa_2026.apply(lambda x: calcular_media_gols_10(x['home_team'], x['date'], df_treino), axis=1)
df_copa_2026['away_form'] = df_copa_2026.apply(lambda x: calcular_media_gols_10(x['away_team'], x['date'], df_treino), axis=1)

In [3]:
# =============================================
# Engenharia de Features: Codificar times e torneios
# =============================================

import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# 1. IDs de Times e Torneios
le_teams = LabelEncoder().fit(pd.concat([df_treino['home_team'], df_treino['away_team'], df_copa_2026['home_team'], df_copa_2026['away_team']]))
le_tourn = LabelEncoder().fit(pd.concat([df_treino['tournament'], df_copa_2026['tournament']]))

for df in [df_treino, df_copa_2026]:
    df['home_id'] = le_teams.transform(df['home_team'])
    df['away_id'] = le_teams.transform(df['away_team'])
    df['tournament_id'] = le_tourn.transform(df['tournament'])
    df['neutral'] = df['neutral'].astype(int)

# 2. Treino Final com as 6 features
features = ['home_id', 'away_id', 'tournament_id', 'neutral', 'home_form', 'away_form']
print("Treinando a IA oficial com versão lite (Leve)...")
model_home = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_split=5, 
    random_state=42).fit(df_treino[features], df_treino['home_score'])

model_away = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_split=5, 
    random_state=42).fit(df_treino[features], df_treino['away_score'])

# ============================
# Salvar com Compressão Máxima
# ============================
print("Compactando e salvando os modelos treinados...")
joblib.dump(model_home, 'model_home.joblib', compress=3)
joblib.dump(model_away, 'model_away.joblib', compress=3)
joblib.dump(le_teams, 'le_teams.joblib', compress=3)
joblib.dump(le_tourn, 'le_tourn.joblib', compress=3)

print("Arquivos compactados com sucesso! Prontos para subir no GitHub com menos de 2MB.")
# ==============================================
# Salvar e carregar modelos com joblib
# ==============================================

def salvar_modelos(model_home_path='model_home.joblib', model_away_path='model_away.joblib', le_teams_path='le_teams.joblib', le_tourn_path='le_tourn.joblib'):
    dump(model_home, model_home_path)
    dump(model_away, model_away_path)
    dump(le_teams, le_teams_path)
    dump(le_tourn, le_tourn_path)
    print(f"Modelos salvos:\n - {model_home_path}\n - {model_away_path}\n - {le_teams_path}\n - {le_tourn_path}")


def carregar_modelos(model_home_path='model_home.joblib', model_away_path='model_away.joblib', le_teams_path='le_teams.joblib', le_tourn_path='le_tourn.joblib'):
    global model_home, model_away, le_teams, le_tourn
    model_home = load(model_home_path)
    model_away = load(model_away_path)
    le_teams = load(le_teams_path)
    le_tourn = load(le_tourn_path)
    print(f"Modelos carregados de:\n - {model_home_path}\n - {model_away_path}\n - {le_teams_path}\n - {le_tourn_path}")
    return model_home, model_away, le_teams, le_tourn

# Salvar os modelos treinados em disco imediatamente após o treino
salvar_modelos()

Treinando a IA oficial com versão lite (Leve)...
Compactando e salvando os modelos treinados...
Arquivos compactados com sucesso! Prontos para subir no GitHub com menos de 2MB.
Modelos salvos:
 - model_home.joblib
 - model_away.joblib
 - le_teams.joblib
 - le_tourn.joblib


In [4]:
# =================================================================
# Mostra o erro médio da IA usando uma parte dos dados do passado
# =================================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# 1. Separar uma parte dos dados do passado para TESTE (ex: 20% dos jogos)
X_train, X_test, y_train_home, y_test_home, y_train_away, y_test_away = train_test_split(
    df_treino[features], df_treino['home_score'], df_treino['away_score'], 
    test_size=0.2, random_state=42
)

# 2. Treinar um modelo temporário de teste
model_test_home = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train_home)
model_test_away = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train_away)

# 3. Fazer previsões nos jogos que a IA NUNCA VIU
pred_home = model_test_home.predict(X_test)
pred_away = model_test_away.predict(X_test)

# 4. Calcular o Erro Médio de Gols
erro_home = mean_absolute_error(y_test_home, pred_home)
erro_away = mean_absolute_error(y_test_away, pred_away)

print("--- RELATÓRIO DE QUALIDADE DA IA ---")
print(f"Erro médio nos gols do Mandante: {round(erro_home, 2)} gols")
print(f"Erro médio nos gols do Visitante: {round(erro_away, 2)} gols")

--- RELATÓRIO DE QUALIDADE DA IA ---
Erro médio nos gols do Mandante: 1.16 gols
Erro médio nos gols do Visitante: 0.97 gols


In [5]:
# ==============================================
# Função para prever o resultado de um confronto
# ==============================================

def prever_confronto(time_casa, time_fora, torneio='FIFA World Cup', neutral=True):
    id_casa = le_teams.transform([time_casa])[0]
    id_fora = le_teams.transform([time_fora])[0]
    id_torneio = le_tourn.transform([torneio])[0]
    forma_casa = calcular_media_gols_10(time_casa, pd.Timestamp.now(), df_treino)
    forma_fora = calcular_media_gols_10(time_fora, pd.Timestamp.now(), df_treino)
    
    dados_jogo = pd.DataFrame([[id_casa, id_fora, id_torneio, int(neutral), forma_casa, forma_fora]], 
                              columns=['home_id', 'away_id', 'tournament_id', 'neutral', 'home_form', 'away_form'])
    
    return round(model_home.predict(dados_jogo)[0], 2), round(model_away.predict(dados_jogo)[0], 2)


def prever_com_joblib(time_casa, time_fora, torneio='FIFA World Cup', neutral=True,
                     model_home_path='model_home.joblib', model_away_path='model_away.joblib',
                     le_teams_path='le_teams.joblib', le_tourn_path='le_tourn.joblib'):
    carregar_modelos(model_home_path, model_away_path, le_teams_path, le_tourn_path)
    gols_casa, gols_fora = prever_confronto(time_casa, time_fora, torneio, neutral)
    print(f"Previsão: {time_casa} {gols_casa} x {gols_fora} {time_fora}")
    return gols_casa, gols_fora

# Exemplo de uso:
# salvar_modelos()  # salvar uma vez após treinar
prever_com_joblib("Portugal", "Nigeria")


Modelos carregados de:
 - model_home.joblib
 - model_away.joblib
 - le_teams.joblib
 - le_tourn.joblib
Previsão: Portugal 1.39 x 1.27 Nigeria


(np.float64(1.39), np.float64(1.27))